**Residente:** *Brener Gomes dos Santos*

**Data:** *28/11/25*

In [47]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error

In [48]:
!pip install ucimlrepo

In [49]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
abalone = fetch_ucirepo(id=1)

# data (as pandas dataframes)
X = abalone.data.features
y = abalone.data.targets

# metadata
print(abalone.metadata)

# variable information
print(abalone.variables)


{'uci_id': 1, 'name': 'Abalone', 'repository_url': 'https://archive.ics.uci.edu/dataset/1/abalone', 'data_url': 'https://archive.ics.uci.edu/static/public/1/data.csv', 'abstract': 'Predict the age of abalone from physical measurements', 'area': 'Biology', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Tabular'], 'num_instances': 4177, 'num_features': 8, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': [], 'target_col': ['Rings'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1994, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C55C7W', 'creators': ['Warwick Nash', 'Tracy Sellers', 'Simon Talbot', 'Andrew Cawthorn', 'Wes Ford'], 'intro_paper': None, 'additional_info': {'summary': 'Predicting the age of abalone from physical measurements.  The age of abalone is determined by cutting the shell through the cone, staining it, and counting the number of rings through a microscope -- 

In [50]:
y

,Rings
0,15
1,7
2,9
3,10
4,7
...,...
4172,11
4173,10
4174,9
4175,10


In [51]:
df = pd.concat((X,y),axis=1)

In [52]:
df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [53]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [54]:
model = ols('Rings ~ (Length+Height+Viscera_weight):C(Sex)',data=df).fit()
#model = ols('Rings ~ Length+Height:Viscera_weight',data=df).fit()

In [55]:
sm.stats.anova_lm(model, typ=2)

,sum_sq,df,F,PR(>F)
Length:C(Sex),1165.169542,3.0,59.610786,9.971382e-38
Height:C(Sex),1355.129851,3.0,69.329272,9.539670e-44
Viscera_weight:C(Sex),640.705886,3.0,32.778905,6.196963e-21
Residual,27149.792687,4167.0,NaN,NaN


# Exercício

Usando os dados conhecidos do molusco abalone. Encontre um modelo (relação entre variáveis dependentes e indenpendente) em que todas as relações são estatisticamente relevantes e que reduza o resíduo com menos termos.

[Exemplo](https://www.statology.org/two-way-anova-python/)



In [56]:
df = pd.get_dummies(df, columns=["Sex"], drop_first=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Length          4177 non-null   float64
 1   Diameter        4177 non-null   float64
 2   Height          4177 non-null   float64
 3   Whole_weight    4177 non-null   float64
 4   Shucked_weight  4177 non-null   float64
 5   Viscera_weight  4177 non-null   float64
 6   Shell_weight    4177 non-null   float64
 7   Rings           4177 non-null   int64  
 8   Sex_I           4177 non-null   bool   
 9   Sex_M           4177 non-null   bool   
dtypes: bool(2), float64(7), int64(1)
memory usage: 269.3 KB


In [57]:
y = df["Rings"]
X = df.drop(columns=["Rings"])

X = X.select_dtypes(include=[np.number])

X_const = sm.add_constant(X)


In [58]:
full_model = sm.OLS(y, X_const).fit()
print(full_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Rings   R-squared:                       0.528
Model:                            OLS   Adj. R-squared:                  0.527
Method:                 Least Squares   F-statistic:                     665.2
Date:                Sat, 29 Nov 2025   Prob (F-statistic):               0.00
Time:                        01:08:52   Log-Likelihood:                -9250.0
No. Observations:                4177   AIC:                         1.852e+04
Df Residuals:                    4169   BIC:                         1.857e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              2.9852      0.269     11.

In [59]:
def backward_elimination(X, y, sig=0.05):
    Xb = sm.add_constant(X)
    model = sm.OLS(y, Xb).fit()

    while True:
        pvals = model.pvalues.drop("const", errors="ignore")
        worst_p = pvals.max()

        if worst_p > sig:
            worst_var = pvals.idxmax()
            print(f"Removendo: {worst_var}  (p={worst_p:.4f})")
            X = X.drop(columns=[worst_var])
            Xb = sm.add_constant(X)
            model = sm.OLS(y, Xb).fit()
        else:
            break

    return model, X

In [60]:
model_be, X_selected = backward_elimination(X, y)

print("\nVariáveis mantidas no modelo final:")
print(X_selected.columns.tolist())

print("\nModelo final:")
print(model_be.summary())

Removendo: Length  (p=0.3891)

Variáveis mantidas no modelo final:
['Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']

Modelo final:
                            OLS Regression Results                            
Dep. Variable:                  Rings   R-squared:                       0.528
Model:                            OLS   Adj. R-squared:                  0.527
Method:                 Least Squares   F-statistic:                     776.0
Date:                Sat, 29 Nov 2025   Prob (F-statistic):               0.00
Time:                        01:08:53   Log-Likelihood:                -9250.3
No. Observations:                4177   AIC:                         1.851e+04
Df Residuals:                    4170   BIC:                         1.856e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                     coef    std err 

In [61]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lasso = LassoCV(cv=5, random_state=0).fit(X_scaled, y)
coef = pd.Series(lasso.coef_, index=X.columns)

print("Alpha escolhido:", lasso.alpha_)
print("\nCoeficientes ≠ 0:")
print(coef[coef != 0])

Alpha escolhido: 0.002023162577241302

Coeficientes ≠ 0:
Length           -0.044802
Diameter          1.182383
Height            0.492209
Whole_weight      4.245707
Shucked_weight   -4.360811
Viscera_weight   -0.993895
Shell_weight      1.283493
dtype: float64
